# 04 EDA: daily onsets and most frequent messages

Regenerates the exploratory figures from the fault episodes rather than raw log entries, so that recovery messages are not counted as events and zero-fault days are retained. Days out of service for annual maintenance (1 to 29 May 2025) are masked.

Input `build/fault_episodes_v4.csv`; outputs in `OUTPUT/v4/`. Figure 3 of the paper, with descriptive names for the messages, is drawn by `FIG3_NAMED.py`.

Paper: Section 3.1; Figures 2 and 3.

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

CFG = dict(episodes="build/fault_episodes_v4.csv", outdir="OUTPUT/v4", gap_min_days=7, top_n=12)
os.makedirs(CFG["outdir"], exist_ok=True)

plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.7, "figure.dpi": 110,
})
BLUE, RED = "#1f6fb4", "#c0392b"

ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
s = (ep.set_index("onset").resample("D").size().asfreq("D").fillna(0).astype(float))
zero = s.eq(0)
blk = (zero != zero.shift()).cumsum()
gaps = []
for _, idx in s.groupby(blk).groups.items():
    idx = pd.DatetimeIndex(idx)
    if zero.loc[idx].all() and len(idx) >= CFG["gap_min_days"]:
        s.loc[idx] = np.nan
        gaps.append((idx.min(), idx.max()))
print(f"{len(ep)} episodes, {len(s)} days, {int(s.notna().sum())} observed")

## Figure 2: daily fault onsets

In [ ]:
# ---- Figure 2: daily episode counts ---------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
first = True
for a, b in gaps:
    ax.axvspan(a, b, color="0.90", lw=0, zorder=0)
    mid = a + (b - a) / 2
    ax.text(mid, 0.5, "Out of service\n(annual maintenance)", transform=ax.get_xaxis_transform(), rotation=90, ha="center", va="center", fontsize=10, color="0.3")
ax.plot(s.index, s.values, color="0.25", lw=1.0, zorder=4)
peak = s.idxmax()
ax.annotate(f"{int(s.max())} on {peak:%d %b %Y}", xy=(peak, s.max()), xytext=(12, -6), textcoords="offset points", fontsize=9, color="0.3")
ax.set_xlim(s.index[0], s.index[-1])
ax.set_ylim(0, s.max() * 1.12)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.grid(axis="y", color="0.85", lw=0.5)
ax.set_ylabel("Fault onsets per day")
plt.tight_layout()
plt.savefig(f"{CFG['outdir']}/daily_onsets.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{CFG['outdir']}/daily_onsets.pdf", bbox_inches="tight")
plt.show()

## Figure 3: most frequent fault messages

In [ ]:
# ---- Figure 3: most frequent fault labels ---------------------------
vc = ep["fault"].value_counts().head(CFG["top_n"])[::-1]
share = vc / len(ep) * 100
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(vc)), vc.values, color="0.80", zorder=2)
ax.barh(range(len(vc)), vc.values, color=BLUE, zorder=3,
        height=0.0)  # keeps colour cycle stable if restyled
for i, (v, p) in enumerate(zip(vc.values, share.values)):
    ax.text(v + len(ep) * 0.004, i, f"{v:,} ({p:.1f}%)", va="center", fontsize=8, color="0.1")
ax.set_yticks(range(len(vc)))
ax.set_yticklabels([t.title() for t in vc.index], fontsize=8)
ax.set_xlabel("Fault episodes")
ax.set_xlim(0, vc.max() * 1.18)
ax.grid(axis="x", color="0.85", lw=0.5)
plt.tight_layout()
plt.savefig(f"{CFG['outdir']}/top_faults.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{CFG['outdir']}/top_faults.pdf", bbox_inches="tight")
plt.show()

___
**END**